# Dataset Localization Overview

This notebook scans `/playpen-ssd/smerrill/deception2/Dataset`, previews one localization JSON per dataset/model, and computes simple word-count summaries.

In [22]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import JSON, Markdown, display

pd.set_option("display.max_colwidth", 160)

DATASET_ROOT = Path("/playpen-ssd/smerrill/deception2/Dataset")
WORD_RE = re.compile(r"\b\w+\b")


def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(WORD_RE.findall(text))


def localization_files_for(dataset_dir: Path) -> list[Path]:
    return sorted((dataset_dir / "localization").glob("*.json"))


entries = []
for dataset_dir in sorted(DATASET_ROOT.glob("*/*")):
    loc_dir = dataset_dir / "localization"
    if not loc_dir.is_dir():
        continue
    files = localization_files_for(dataset_dir)
    if not files:
        continue
    entries.append(
        {
            "dataset": dataset_dir.parent.name,
            "model": dataset_dir.name,
            "dataset_dir": dataset_dir,
            "localization_dir": loc_dir,
            "num_files": len(files),
            "sample_file": files[0],
        }
    )

entries_df = pd.DataFrame(
    [
        {
            "dataset": entry["dataset"],
            "model": entry["model"],
            "num_localization_files": entry["num_files"],
            "sample_file": str(entry["sample_file"]),
        }
        for entry in entries
    ]
).sort_values(["dataset", "model"], ignore_index=True)

entries_df

,dataset,model,num_localization_files,sample_file
0,AdvisorAudit,DeepSeek-R1-Distill-Qwen-7B,6000,/playpen-ssd/smerrill/deception2/Dataset/AdvisorAudit/DeepSeek-R1-Distill-Qwen-7B/localization/sentence_localization_2026-03-05_gpu_2_state_0_sample_24.json
1,BS,DeepSeek-R1-Distill-Qwen-14B,6124,/playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-14B/localization/sentence_localization_2026-02-10_gpu_2_state_0_sample_0.json
2,BS,DeepSeek-R1-Distill-Qwen-7B,6000,/playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B/localization/sentence_localization_2026-02-06_gpu_2_state_0_sample_0.json
3,Gridworld,DeepSeek-R1-Distill-Qwen-7B,7479,/playpen-ssd/smerrill/deception2/Dataset/Gridworld/DeepSeek-R1-Distill-Qwen-7B/localization/sentence_localization_2026-02-18_gpu_2_state_0_sample_0.json


In [23]:
def mean_or_na(values: list[int | float]) -> float:
    return float(sum(values) / len(values)) if values else float("nan")


def shorten(text: str, max_chars: int = 240) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\n", " ").strip()
    return text if len(text) <= max_chars else text[: max_chars - 3] + "..."


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def history_word_counts(record: dict) -> list[int]:
    history = record.get("history") or []
    return [word_count(item.get("sentence_text", "")) for item in history]


def sample_file_summary(sample_file: Path) -> dict:
    record = load_json(sample_file)
    history = record.get("history") or []
    history_counts = history_word_counts(record)
    return {
        "example_id": record.get("example_id"),
        "game": record.get("game"),
        "raw_text_words": word_count(record.get("raw_text", "")),
        "prompt_words": word_count(record.get("prompt", "")),
        "history_sentences": len(history),
        "avg_history_sentence_words": mean_or_na(history_counts),
        "min_history_sentence_words": min(history_counts) if history_counts else None,
        "max_history_sentence_words": max(history_counts) if history_counts else None,
        "full_score_sentence_words": word_count((record.get("full_score") or {}).get("sentence_text", "")),
    }


sample_stats_df = pd.DataFrame(
    [
        {
            "dataset": entry["dataset"],
            "model": entry["model"],
            **sample_file_summary(entry["sample_file"]),
        }
        for entry in entries
    ]
).sort_values(["dataset", "model"], ignore_index=True)

sample_stats_df.round(2)

,dataset,model,example_id,game,raw_text_words,prompt_words,history_sentences,avg_history_sentence_words,min_history_sentence_words,max_history_sentence_words,full_score_sentence_words
0,AdvisorAudit,DeepSeek-R1-Distill-Qwen-7B,2026-03-05/gpu_2/state_0/sample_24,advisor_audit,498,288,16,10.19,0,24,0
1,BS,DeepSeek-R1-Distill-Qwen-14B,2026-02-10/gpu_2/state_0/sample_0,bs,808,244,16,13.25,1,20,1
2,BS,DeepSeek-R1-Distill-Qwen-7B,2026-02-06/gpu_2/state_0/sample_0,bs,626,244,16,13.44,1,29,1
3,Gridworld,DeepSeek-R1-Distill-Qwen-7B,2026-02-18/gpu_2/state_0/sample_0,gridworld,440,276,15,15.87,1,30,1


In [ ]:
for entry in entries:
    sample_file = entry["sample_file"]
    record = load_json(sample_file)
    history = record.get("history") or []

    display(Markdown(f"## {entry['dataset']} / {entry['model']}"))
    display(Markdown(f"Sample file: `{sample_file}`"))
    display(pd.DataFrame([sample_file_summary(sample_file)]).round(2))

    preview = {
        "example_id": record.get("example_id"),
        "game": record.get("game"),
        "top_level_keys": sorted(record.keys()),
        "raw_text_preview": shorten(record.get("raw_text", ""), max_chars=500),
        "prompt_preview": shorten(record.get("prompt", ""), max_chars=500),
        "history_len": len(history),
        "first_history_entry": history[0] if history else None,
    }
    #display(JSON(preview, expanded=False))

    if history:
        history_preview = pd.DataFrame(
            [
                {
                    "sentence_idx_inclusive": item.get("sentence_idx_inclusive"),
                    "deception_rate": item.get("deception_rate"),
                    "word_count": word_count(item.get("sentence_text", "")),
                    "sentence_text": shorten(item.get("sentence_text", ""), max_chars=140),
                }
                for item in history[:5]
            ]
        )
        display(history_preview)


In [ ]:
def dataset_word_stats(entry: dict) -> dict:
    raw_text_counts = []
    prompt_counts = []
    history_sentence_counts = []
    history_lengths = []
    history_file_averages = []

    for sample_file in localization_files_for(entry["dataset_dir"]):
        record = load_json(sample_file)
        raw_text_counts.append(word_count(record.get("raw_text", "")))
        prompt_counts.append(word_count(record.get("prompt", "")))

        history_counts = history_word_counts(record)
        history_sentence_counts.extend(history_counts)
        history_lengths.append(len(history_counts))
        if history_counts:
            history_file_averages.append(mean_or_na(history_counts))

    return {
        "dataset": entry["dataset"],
        "model": entry["model"],
        "num_localization_files": entry["num_files"],
        "avg_raw_text_words": mean_or_na(raw_text_counts),
        "avg_prompt_words": mean_or_na(prompt_counts),
        "avg_history_sentence_words": mean_or_na(history_sentence_counts),
        "avg_history_sentences_per_file": mean_or_na(history_lengths),
        "avg_history_words_per_file": mean_or_na(history_file_averages),
    }


dataset_stats_df = pd.DataFrame([dataset_word_stats(entry) for entry in entries]).sort_values(
    ["dataset", "model"], ignore_index=True
)

dataset_stats_df.round(2)

In [ ]:
def pick_sample_file(dataset_name: str, preferred_model: str | None = None) -> Path:
    matches = [entry for entry in entries if entry["dataset"] == dataset_name]
    if preferred_model is not None:
        for entry in matches:
            if entry["model"] == preferred_model:
                return entry["sample_file"]
    if not matches:
        raise ValueError(f"No dataset entry found for {dataset_name}")
    return matches[0]["sample_file"]


target_files = {
    "BS": pick_sample_file("BS", preferred_model="DeepSeek-R1-Distill-Qwen-7B"),
    "Gridworld": pick_sample_file("Gridworld", preferred_model="DeepSeek-R1-Distill-Qwen-7B"),
    "AdvisorAudit": pick_sample_file("AdvisorAudit", preferred_model="DeepSeek-R1-Distill-Qwen-7B"),
}

for dataset_name, target_file in target_files.items():
    data = load_json(target_file)
    print(f"\n=== {dataset_name} ===")
    print(target_file)
    print(data["raw_text"])
